# Valutazione risposte RAG

Esegue i casi in `eval/retrieval.json` contro `POST /search`, confronta risposta e riferimento, assegna un voto LLM da 1 a 3 e salva un Excel. Richiede API locale attiva, `.env` configurato e servizi Azure raggiungibili. Il voto è una stima automatica: controlla i casi con voto 1 o 2.

In [ ]:
import json
import os
import time
from difflib import SequenceMatcher
from pathlib import Path

import pandas as pd
import requests
from dotenv import load_dotenv
from openai import AzureOpenAI

ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / "eval" / "retrieval.json").exists()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Impossibile trovare eval/retrieval.json risalendo dalla cartella corrente")

load_dotenv(ROOT / ".env")
RAG_API_URL = os.getenv("RAG_API_URL", "http://127.0.0.1:8000").rstrip("/")
RAG_API_KEY = os.environ["API_KEY"]
JUDGE_DEPLOYMENT = os.getenv("OPENAI_DEPLOYMENT_NAME", "gpt-4o")

judge_client = AzureOpenAI(
    azure_endpoint=os.environ["OPENAI_URL"],
    api_key=os.environ["OPENAI_KEY"],
    api_version=os.getenv("OPENAI_API_VERSION", "2024-10-21"),
)
print(f"Dataset: {ROOT / 'eval' / 'retrieval.json'}")
print(f"Endpoint RAG: {RAG_API_URL}/search")

## 2. Casi di test

Carica le domande e le risposte di riferimento dal dataset già preparato.

In [ ]:
cases = json.loads((ROOT / "eval" / "retrieval.json").read_text(encoding="utf-8"))
tests = pd.DataFrame(cases)
tests.insert(0, "id_test", range(1, len(tests) + 1))
tests[["id_test", "question", "reference_answer"]].head()

## 3. Chiamata endpoint e raccolta risposte

L’API del progetto usa `POST /search` (non `/chat`); misuro anche il tempo di risposta.

In [ ]:
TOP_K = 30


def build_context(sources):
    formatted_sources = []
    for position, source in enumerate(sources, start=1):
        metadata = source.get("metadata") or {}
        if not isinstance(metadata, dict):
            metadata = {}
        document = (
            metadata.get("document_title")
            or metadata.get("source_file")
            or metadata.get("source")
            or "Documento sconosciuto"
        )
        section = metadata.get("section_path")
        page = metadata.get("page")
        citation = f"[Fonte {position}; documento: {document}"
        if section:
            citation += f"; sezione: {section}"
        if page is not None:
            citation += f"; pagina: {page}"
        formatted_sources.append(f"{citation}]\n{source.get('content', '')}")
    return "\n\n".join(formatted_sources) or "Nessuna fonte pertinente è stata trovata."


def verify_api_top_k():
    response = requests.get(f"{RAG_API_URL}/openapi.json", timeout=10)
    response.raise_for_status()
    api_max = response.json()["components"]["schemas"]["SearchRequest"]["properties"]["top_k"]["maximum"]
    if api_max < TOP_K:
        raise RuntimeError(
            f"L’API attiva accetta top_k fino a {api_max}, ma il notebook richiede {TOP_K}. "
            "Riavvia Uvicorn per caricare la nuova configurazione."
        )


def ask_rag(question):
    started = time.perf_counter()
    response = requests.post(
        f"{RAG_API_URL}/search",
        headers={"Authorization": f"Bearer {RAG_API_KEY}"},
        json={"query": question, "top_k": TOP_K},
        timeout=120,
    )
    response.raise_for_status()
    return response.json(), time.perf_counter() - started


verify_api_top_k()
answers = []
retrieved_chunks = []
for row in tests.itertuples(index=False):
    try:
        payload, latency = ask_rag(row.question)
        answer = payload.get("answer") or ""
        api_status = "ok" if answer.strip() else "empty_answer"
        api_error = "" if answer.strip() else "L’endpoint ha restituito una risposta vuota."
        sources = payload.get("sources") or []
    except requests.RequestException as exc:
        answer, latency, sources = "", None, []
        api_status, api_error = "request_error", str(exc)

    context = build_context(sources) if api_status != "request_error" else ""
    answers.append(
        {
            "id_test": row.id_test,
            "risposta_modello": answer,
            "contesto_llm": context,
            "latenza_s": round(latency, 2) if latency is not None else None,
            "api_status": api_status,
            "api_error": api_error,
        }
    )
    for rank, source in enumerate(sources, start=1):
        metadata = source.get("metadata") or {}
        if not isinstance(metadata, dict):
            metadata = {}
        retrieved_chunks.append(
            {
                "id_test": row.id_test,
                "prompt": row.question,
                "rank": rank,
                "documento": metadata.get("document_title") or metadata.get("source_file") or metadata.get("source"),
                "sezione": metadata.get("section_path"),
                "pagina": metadata.get("page"),
                "similarity_score": source.get("score"),
                "chunk_text": source.get("content", ""),
            }
        )

results = tests.merge(pd.DataFrame(answers), on="id_test")
results[["id_test", "api_status", "question", "risposta_modello", "latenza_s"]].head()

### Ispeziona chunk e contesto completo

Imposta l’ID del test da controllare. Il contesto è assemblato dalla stessa funzione usata dal RAG prima della chiamata LLM.

In [ ]:
TEST_ID_TO_INSPECT = 1  # Cambia ID per vedere un altro caso
selected = results.loc[results["id_test"] == TEST_ID_TO_INSPECT].iloc[0]
selected_chunks = pd.DataFrame(
    [chunk for chunk in retrieved_chunks if chunk["id_test"] == TEST_ID_TO_INSPECT],
    columns=["rank", "documento", "sezione", "pagina", "similarity_score", "chunk_text"],
)
print(f"Test {TEST_ID_TO_INSPECT}: {selected['question']}")
print(f"Stato API: {selected['api_status']}")
if selected["api_error"]:
    print(f"Errore API: {selected['api_error']}")
display(selected_chunks)
if selected["api_status"] == "request_error":
    print("Contesto non disponibile: la richiesta API non è riuscita.")
else:
    if selected_chunks.empty:
        print("La richiesta è riuscita, ma l’API non ha restituito chunk.")
    print("\nContesto completo passato al modello:\n")
    print(selected["contesto_llm"])

## 4. Confronto testuale

La similarità lessicale è solo un indicatore: non determina da sola il voto, perché una parafrasi corretta può avere parole diverse.

In [ ]:
import re


def normalize(text):
    return re.sub(r"[^\w\s]", " ", str(text).casefold()).split()


results["similarita"] = results.apply(
    lambda row: round(
        SequenceMatcher(None, normalize(row["reference_answer"]), normalize(row["risposta_modello"])).ratio(), 3
    ),
    axis=1,
)
results[["id_test", "similarita"]].head()

## 5. Voto e discrepanze

Il giudice confronta risposta, riferimento ed evidenza: 1 = errata/non supportata, 2 = parziale o con omissioni, 3 = corretta e sufficientemente completa.

In [ ]:
def judge(row):
    if not str(row.risposta_modello).strip():
        reason = row.api_error or "L’endpoint non ha restituito una risposta."
        return 1, reason

    payload = {
        "question": row.question,
        "reference_answer": row.reference_answer,
        "evidence": row.evidence,
        "rag_answer": row.risposta_modello,
    }
    completion = judge_client.chat.completions.create(
        model=JUDGE_DEPLOYMENT,
        temperature=0,
        response_format={"type": "json_object"},
        messages=[
            {
                "role": "system",
                "content": (
                    "Valuta la risposta RAG usando riferimento ed evidenza come verità attesa. "
                    "Assegna 1 se è errata, contraddetta o non supportata; 2 se è parziale o omette "
                    "un dettaglio importante; 3 solo se risponde correttamente e in modo sufficientemente "
                    "completo al nucleo della domanda. Non penalizzare parafrasi corrette. Se la risposta "
                    "dice di non avere informazioni o che la fonte non è disponibile mentre riferimento ed "
                    "evidenza contengono la risposta, assegna 1. Non assegnare 3 a una risposta che non "
                    "risponde alla domanda. Restituisci solo JSON con score (intero 1-3) e discrepancy "
                    "(spiega in modo conciso cosa è sbagliato o manca; se nulla, scrivi "
                    "'Nessuna discrepanza rilevante.')."
                ),
            },
            {"role": "user", "content": json.dumps(payload, ensure_ascii=False)},
        ],
    )
    judgment = json.loads(completion.choices[0].message.content)
    score = int(judgment["score"])
    if score not in (1, 2, 3):
        raise ValueError(f"Punteggio giudice non valido: {score}")
    return score, str(judgment["discrepancy"])


judgments = results.apply(judge, axis=1)
results[["punteggio", "discrepanza"]] = pd.DataFrame(judgments.tolist(), index=results.index)
results[["id_test", "punteggio", "discrepanza"]].head()

## 6. Report Excel

In [ ]:
report = results.rename(columns={"question": "prompt", "reference_answer": "riferimento"})
report["note"] = report["latenza_s"].map(
    lambda seconds: f"Latenza endpoint: {seconds:.2f} s" if pd.notna(seconds) else ""
)
report = report[
    [
        "id_test", "prompt", "riferimento", "risposta_modello", "similarita",
        "punteggio", "discrepanza", "api_status", "api_error", "note",
    ]
]
chunk_columns = [
    "id_test", "prompt", "rank", "documento", "sezione", "pagina", "similarity_score", "chunk_text"
]
chunks_report = pd.DataFrame(retrieved_chunks, columns=chunk_columns)

context_rows = []
for row in results.itertuples(index=False):
    context = row.contesto_llm
    parts = [context[i:i + 30000] for i in range(0, len(context), 30000)] or [""]
    for part_number, part in enumerate(parts, start=1):
        context_rows.append(
            {
                "id_test": row.id_test,
                "prompt": row.question,
                "parte": part_number,
                "parti_totali": len(parts),
                "contesto": part,
            }
        )
contexts_report = pd.DataFrame(
    context_rows,
    columns=["id_test", "prompt", "parte", "parti_totali", "contesto"],
)

output_path = ROOT / "output" / "chat_answer_evaluation.xlsx"
output_path.parent.mkdir(parents=True, exist_ok=True)
with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    report.to_excel(writer, index=False, sheet_name="Risultati")
    chunks_report.to_excel(writer, index=False, sheet_name="Chunk recuperati")
    contexts_report.to_excel(writer, index=False, sheet_name="Contesto LLM")
print(f"Excel creato: {output_path}")
print(f"Punteggio medio: {report['punteggio'].mean():.2f}/3")
print(f"Chunk registrati: {len(chunks_report)}")

## 7. Casi da rivedere

Mostra prima i casi con voto basso e similarità testuale minore.

In [ ]:
critical = results.loc[results["punteggio"] <= 2].sort_values("similarita")
display(
    critical[["id_test", "question", "reference_answer", "risposta_modello", "similarita", "punteggio", "discrepanza"]]
)